In [1]:
import pysam
import numpy as np
import pandas as pd
import itertools
import re

from Bio import SeqIO
from Bio.Seq import Seq
from tqdm import tqdm

In [2]:
def get_read_flag(read, mapq_thre =20):
    is_mapped  = (not read.is_unmapped)
    is_primary = (not read.is_secondary) and (not read.is_supplementary)
    is_quality = (not read.is_qcfail) and read.mapq >= mapq_thre
    is_fully_align = all(op in (0, 7, 8) for op, length in read.cigar)
    return  is_mapped and is_primary and is_quality and is_fully_align


def gen_read_pair(bam_file, contig_id):
    bam = pysam.AlignmentFile(bam_file, "rb")
    
    for _, reads in itertools.groupby(bam.fetch(contig=contig_id), key=lambda x: x.query_name):
        reads = list(reads)
        if len(reads) == 2:
            read1, read2 = reads
            read1_flag = get_read_flag(read1)
            read2_flag = get_read_flag(read2)

            if read1_flag and read2_flag and read1.reference_name == read2.reference_name:
                if read1.is_read1 and read2.is_read2:
                    yield read1, read2
                else:
                    yield read2, read1

In [7]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"

ref_fasta = working_path + "/data/ref/GRCh38_full_analysis_set_plus_decoy_hla.fa"
bam_file  = working_path + "/data/TBS/20000.sorted.mdup.bam"

In [8]:
ref_dict  = SeqIO.to_dict(SeqIO.parse(ref_fasta, "fasta"))

In [31]:
read_len = 151
pos_err_arr  = np.zeros((2, read_len))

In [32]:
pos_err_arr

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0.,

In [40]:
x = gen_read_pair(bam_file, "chr21")

In [57]:
read1, read2 = next(x)

In [71]:
for read1, read2 in gen_read_pair(bam_file, "chr21"):
    if read1.reference_start < read2.reference_start and read2.reference_end > read1.reference_end:
        if read2.reference_start - read1.reference_end <= 0:
            break
    

In [72]:
[read1.reference_start, read1.reference_end, read2.reference_start, read2.reference_end]

[13664831, 13664982, 13664885, 13665036]

In [73]:
overlap_len = read2.reference_start - read1.reference_end

In [74]:
overlap_len

-97

In [75]:
read1.seq

'GATATATTTTTATATTGTTAGATATTATTATATAGATATTGTTATGTGGATATAAGGATATATAGATATTGTTATATGGATATATATATTGTTATATGGAGATATTATTATGTAGATATATGAATATATTATTATATGGATATAGTATTAT'

In [76]:
read2.seq

'AGGATATATAGATATTGTTATATGGATATATATATTGTTATATGGAGATATTATTATGTAGATATATGAATATATTATTATATGGATATAGTATTATATAGATATTTTGTTATACGGACATATTATTATATAGATATATTATTACGTTGTT'

In [79]:
read1.seq[overlap_len:]

'AGGATATATAGATATTGTTATATGGATATATATATTGTTATATGGAGATATTATTATGTAGATATATGAATATATTATTATATGGATATAGTATTAT'

In [80]:
read2.seq[:  ]

'AGGATATATAGATATTGTTATATGGATATATATATTGTTATATGGAGATATTAT'

In [86]:
read2.query_sequence

'AGGATATATAGATATTGTTATATGGATATATATATTGTTATATGGAGATATTATTATGTAGATATATGAATATATTATTATATGGATATAGTATTATATAGATATTTTGTTATACGGACATATTATTATATAGATATATTATTACGTTGTT'

In [101]:
read2.tags

[('NM', 0),
 ('MD', '151'),
 ('XB',
  '4z1z1z3z5z1z7z1z1z1z3z1z1x5z2z1zz3z5z5z1z1zz1zz1z5z4z1zz1z1z3z1z1z2zz1z1X3Z1z1zz1zz1z1z3z1z1zz1zz1X1z2zz'),
 ('XC', 0),
 ('AS', 151),
 ('XS', 141),
 ('YS', 'W_G2A'),
 ('MQ', 60),
 ('MC', '151M'),
 ('ms', 5477)]

In [104]:
read1.tags

[('NM', 0),
 ('MD', '151'),
 ('XB',
  '2z1z1zz1zz1z1y3y3z1zz1zz1z1y3z1y2zz7z1z5z1z1y3z5z1z7z1z1z1y3z1z1x5z2z1zz3y5z5z1z1zz1zz1z5z4z1zz1z'),
 ('XC', 0),
 ('AS', 151),
 ('XS', 141),
 ('YS', 'W_C2T'),
 ('MQ', 60),
 ('MC', '151M'),
 ('ms', 5464)]

In [99]:
read1.flag

67

In [105]:
read1.infer_read_length()

151

In [108]:
read2.infer_read_length()

151

In [9]:
for read1, read2 in gen_read_pair(bam_file, "chr21"):
    overlap_len = read2.reference_start - read1.reference_end
    if overlap_len <= 0:
        read1_overlap = read1.seq[-overlap_len:]
        read2_overlap = read2.seq[:overlap_len]
        
    
        
        
    
    print(read1)
    break

A00817:419:HC3JFDSX5:2:2603:23827:5776	115	#20	5101917	60	109M	#20	5101917	0	TAACTCATATAATAACAAACCAACCTTTTAAAAACATTCTATACCCTCAACAAAAACAACCCCTATCTAAAAACCCCCCCAAAAAAACTAACTTCCTCCCCTCCTACCT	array('B', [37, 37, 37, 11, 37, 37, 25, 25, 11, 37, 37, 37, 25, 37, 11, 25, 37, 11, 37, 37, 37, 25, 37, 11, 25, 37, 37, 37, 37, 37, 37, 37, 11, 11, 25, 11, 37, 37, 37, 37, 37, 25, 25, 11, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 25, 37, 37, 37, 37, 25, 37, 37, 25, 37, 37, 37, 37, 37, 37, 37, 37, 25, 37, 37, 37, 37, 11, 37, 11, 37, 37, 25, 25, 37, 37, 37, 25, 37, 37, 11, 37, 37, 37, 37, 37, 37, 37, 25, 37, 11, 37, 37, 11, 37, 37, 37, 37, 25, 37])	[('NM', 1), ('MD', '102G6'), ('XB', '1z6z1z5x4x7zz1z7y1z12z1xy5y3yz10x1zz1z3y15y3'), ('XC', 0), ('AS', 104), ('XS', 89), ('YS', 'C_C2T'), ('MQ', 60), ('MC', '109M'), ('ms', 3434)]


In [38]:
read2.positions

[5101916,
 5101917,
 5101918,
 5101919,
 5101920,
 5101921,
 5101922,
 5101923,
 5101924,
 5101925,
 5101926,
 5101927,
 5101928,
 5101929,
 5101930,
 5101931,
 5101932,
 5101933,
 5101934,
 5101935,
 5101936,
 5101937,
 5101938,
 5101939,
 5101940,
 5101941,
 5101942,
 5101943,
 5101944,
 5101945,
 5101946,
 5101947,
 5101948,
 5101949,
 5101950,
 5101951,
 5101952,
 5101953,
 5101954,
 5101955,
 5101956,
 5101957,
 5101958,
 5101959,
 5101960,
 5101961,
 5101962,
 5101963,
 5101964,
 5101965,
 5101966,
 5101967,
 5101968,
 5101969,
 5101970,
 5101971,
 5101972,
 5101973,
 5101974,
 5101975,
 5101976,
 5101977,
 5101978,
 5101979,
 5101980,
 5101981,
 5101982,
 5101983,
 5101984,
 5101985,
 5101986,
 5101987,
 5101988,
 5101989,
 5101990,
 5101991,
 5101992,
 5101993,
 5101994,
 5101995,
 5101996,
 5101997,
 5101998,
 5101999,
 5102000,
 5102001,
 5102002,
 5102003,
 5102004,
 5102005,
 5102006,
 5102007,
 5102008,
 5102009,
 5102010,
 5102011,
 5102012,
 5102013,
 5102014,
 5102015,


109

In [13]:
read1.aligned_pairs

[(0, 5101916),
 (1, 5101917),
 (2, 5101918),
 (3, 5101919),
 (4, 5101920),
 (5, 5101921),
 (6, 5101922),
 (7, 5101923),
 (8, 5101924),
 (9, 5101925),
 (10, 5101926),
 (11, 5101927),
 (12, 5101928),
 (13, 5101929),
 (14, 5101930),
 (15, 5101931),
 (16, 5101932),
 (17, 5101933),
 (18, 5101934),
 (19, 5101935),
 (20, 5101936),
 (21, 5101937),
 (22, 5101938),
 (23, 5101939),
 (24, 5101940),
 (25, 5101941),
 (26, 5101942),
 (27, 5101943),
 (28, 5101944),
 (29, 5101945),
 (30, 5101946),
 (31, 5101947),
 (32, 5101948),
 (33, 5101949),
 (34, 5101950),
 (35, 5101951),
 (36, 5101952),
 (37, 5101953),
 (38, 5101954),
 (39, 5101955),
 (40, 5101956),
 (41, 5101957),
 (42, 5101958),
 (43, 5101959),
 (44, 5101960),
 (45, 5101961),
 (46, 5101962),
 (47, 5101963),
 (48, 5101964),
 (49, 5101965),
 (50, 5101966),
 (51, 5101967),
 (52, 5101968),
 (53, 5101969),
 (54, 5101970),
 (55, 5101971),
 (56, 5101972),
 (57, 5101973),
 (58, 5101974),
 (59, 5101975),
 (60, 5101976),
 (61, 5101977),
 (62, 5101978),
 (

In [14]:
read2.aligned_pairs

[(0, 5101916),
 (1, 5101917),
 (2, 5101918),
 (3, 5101919),
 (4, 5101920),
 (5, 5101921),
 (6, 5101922),
 (7, 5101923),
 (8, 5101924),
 (9, 5101925),
 (10, 5101926),
 (11, 5101927),
 (12, 5101928),
 (13, 5101929),
 (14, 5101930),
 (15, 5101931),
 (16, 5101932),
 (17, 5101933),
 (18, 5101934),
 (19, 5101935),
 (20, 5101936),
 (21, 5101937),
 (22, 5101938),
 (23, 5101939),
 (24, 5101940),
 (25, 5101941),
 (26, 5101942),
 (27, 5101943),
 (28, 5101944),
 (29, 5101945),
 (30, 5101946),
 (31, 5101947),
 (32, 5101948),
 (33, 5101949),
 (34, 5101950),
 (35, 5101951),
 (36, 5101952),
 (37, 5101953),
 (38, 5101954),
 (39, 5101955),
 (40, 5101956),
 (41, 5101957),
 (42, 5101958),
 (43, 5101959),
 (44, 5101960),
 (45, 5101961),
 (46, 5101962),
 (47, 5101963),
 (48, 5101964),
 (49, 5101965),
 (50, 5101966),
 (51, 5101967),
 (52, 5101968),
 (53, 5101969),
 (54, 5101970),
 (55, 5101971),
 (56, 5101972),
 (57, 5101973),
 (58, 5101974),
 (59, 5101975),
 (60, 5101976),
 (61, 5101977),
 (62, 5101978),
 (

In [23]:
pos_arr

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0.,

In [15]:
read1.seq

'TAACTCATATAATAACAAACCAACCTTTTAAAAACATTCTATACCCTCAACAAAAACAACCCCTATCTAAAAACCCCCCCAAAAAAACTAACTTCCTCCCCTCCTACCT'

In [18]:
read2.seq

'TAACTCATATAATAACAAACCAACCTTTTAAAAACATTCTATACCCTCAACAAAAACAACCCCTATCTAAAAACCACCCCAACAAATCTAACTTCCTCCCCTACTACCA'

In [ ]:
# get read1 and read2
# get the overlapped region
# compare the overlapped region



In [25]:
read1.get_overlap(read2)

TypeError: get_overlap() takes exactly 2 positional arguments (1 given)

In [26]:
read1.aligned_pairs

[(0, 5101916),
 (1, 5101917),
 (2, 5101918),
 (3, 5101919),
 (4, 5101920),
 (5, 5101921),
 (6, 5101922),
 (7, 5101923),
 (8, 5101924),
 (9, 5101925),
 (10, 5101926),
 (11, 5101927),
 (12, 5101928),
 (13, 5101929),
 (14, 5101930),
 (15, 5101931),
 (16, 5101932),
 (17, 5101933),
 (18, 5101934),
 (19, 5101935),
 (20, 5101936),
 (21, 5101937),
 (22, 5101938),
 (23, 5101939),
 (24, 5101940),
 (25, 5101941),
 (26, 5101942),
 (27, 5101943),
 (28, 5101944),
 (29, 5101945),
 (30, 5101946),
 (31, 5101947),
 (32, 5101948),
 (33, 5101949),
 (34, 5101950),
 (35, 5101951),
 (36, 5101952),
 (37, 5101953),
 (38, 5101954),
 (39, 5101955),
 (40, 5101956),
 (41, 5101957),
 (42, 5101958),
 (43, 5101959),
 (44, 5101960),
 (45, 5101961),
 (46, 5101962),
 (47, 5101963),
 (48, 5101964),
 (49, 5101965),
 (50, 5101966),
 (51, 5101967),
 (52, 5101968),
 (53, 5101969),
 (54, 5101970),
 (55, 5101971),
 (56, 5101972),
 (57, 5101973),
 (58, 5101974),
 (59, 5101975),
 (60, 5101976),
 (61, 5101977),
 (62, 5101978),
 (

In [27]:
read2.aligned_pairs

[(0, 5101916),
 (1, 5101917),
 (2, 5101918),
 (3, 5101919),
 (4, 5101920),
 (5, 5101921),
 (6, 5101922),
 (7, 5101923),
 (8, 5101924),
 (9, 5101925),
 (10, 5101926),
 (11, 5101927),
 (12, 5101928),
 (13, 5101929),
 (14, 5101930),
 (15, 5101931),
 (16, 5101932),
 (17, 5101933),
 (18, 5101934),
 (19, 5101935),
 (20, 5101936),
 (21, 5101937),
 (22, 5101938),
 (23, 5101939),
 (24, 5101940),
 (25, 5101941),
 (26, 5101942),
 (27, 5101943),
 (28, 5101944),
 (29, 5101945),
 (30, 5101946),
 (31, 5101947),
 (32, 5101948),
 (33, 5101949),
 (34, 5101950),
 (35, 5101951),
 (36, 5101952),
 (37, 5101953),
 (38, 5101954),
 (39, 5101955),
 (40, 5101956),
 (41, 5101957),
 (42, 5101958),
 (43, 5101959),
 (44, 5101960),
 (45, 5101961),
 (46, 5101962),
 (47, 5101963),
 (48, 5101964),
 (49, 5101965),
 (50, 5101966),
 (51, 5101967),
 (52, 5101968),
 (53, 5101969),
 (54, 5101970),
 (55, 5101971),
 (56, 5101972),
 (57, 5101973),
 (58, 5101974),
 (59, 5101975),
 (60, 5101976),
 (61, 5101977),
 (62, 5101978),
 (

In [29]:
read1.get_overlap()

5101916